In [1]:
!pwd

/content


In [2]:
# Declaring the working library
import os
os.chdir('/content')

In [1]:
# Installing the required library
# !pip install -U uv
# !uv pip install -U scikit-learn

In [2]:
!uv pip show scikit-learn

Using Python 3.12.13 environment at: /usr
Name: scikit-learn
Version: 1.8.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: joblib, numpy, scipy, threadpoolctl
Required-by: esda, fastai, hdbscan, imbalanced-learn, libpysal, librosa, mapclassify, mlxtend, pynndescent, pysal, segregation, sentence-transformers, shap, sklearn-compat, sklearn-pandas, spopt, spreg, tsfresh, umap-learn, yellowbrick


**Forecasts Performance Evaluation**

- **Country: Canada**

In [3]:
import os
os.chdir('/content')

In [5]:
import pandas as pd
import numpy as np
from sklearn import metrics
from sklearn.metrics import mean_squared_log_error

EPSILON = 1e-10

# Helper functions for error calculations
def _error(actual: np.ndarray, predicted: np.ndarray):
    """Simple error"""
    return actual - predicted

def _percentage_error(actual: np.ndarray, predicted: np.ndarray):
    """Percentage error (not multiplied by 100)"""
    return _error(actual, predicted) / (actual + EPSILON)

def _naive_forecasting(actual: np.ndarray, seasonality: int = 1):
    """Naive forecasting method which just repeats previous samples"""
    if len(actual) <= seasonality:
        return np.full_like(actual, np.nan)  # Handle cases where not enough data for naive forecasting
    return np.concatenate((np.full(seasonality, actual[0]), actual[:-seasonality]))

# Function for RMSE
def rmse(actual: np.ndarray, predicted: np.ndarray):
    """Root Mean Squared Error"""
    return np.sqrt(np.mean(np.square(_error(actual, predicted))))

# Function for SMAPE
def smape(actual: np.ndarray, predicted: np.ndarray):
    """Symmetric Mean Absolute Percentage Error"""
    return np.mean(2.0 * np.abs(actual - predicted) / (np.abs(actual) + np.abs(predicted) + EPSILON))

# Function for MDRAE
def mdrae(actual: np.ndarray, predicted: np.ndarray, benchmark: np.ndarray = None):
    """Median Relative Absolute Error"""
    if benchmark is None:
        benchmark = _naive_forecasting(actual)
    abs_error_model = np.abs(predicted - actual)
    abs_error_naive = np.abs(benchmark - actual)
    return np.median(abs_error_model / (abs_error_naive + EPSILON))

# Function for MDAPE
def mdape(actual: np.ndarray, predicted: np.ndarray):
    """Median Absolute Percentage Error"""
    return np.median(np.abs(_percentage_error(actual, predicted))) * 100

# Updated Theil's U1 function
def theils_u1(f: np.ndarray, y: np.ndarray):
    """Theil's U1 statistic"""
    df = pd.DataFrame({'f_i': f, 'y_i': y})
    df['(f_i - y_i)^2'] = np.square(df['f_i'] - df['y_i'])
    df['y_i^2'] = np.square(df['y_i'])
    df['f_i^2'] = np.square(df['f_i'])
    return np.sqrt(np.mean(df['(f_i - y_i)^2'])) / (np.sqrt(np.mean(df['y_i^2'])) + np.sqrt(np.mean(df['f_i^2'])))

# Theil's U2 function
def theils_u2(f: np.ndarray, y: np.ndarray) -> float:
    """Theil's U2 statistic"""
    df = pd.DataFrame({'f_i+1': f.flatten(), 'y_i+1': y.flatten()})
    df['y_i'] = df['y_i+1'].shift(1)
    df['numerator'] = np.square((df['f_i+1'] - df['y_i+1']) / df['y_i'])
    df['denominator'] = np.square((df['y_i+1'] - df['y_i']) / df['y_i'])
    df.dropna(inplace=True)
    return np.sqrt(np.sum(df['numerator']) / np.sum(df['denominator']))

# Function for MASE
def mase(actual: np.ndarray, predicted: np.ndarray, seasonality: int = 1):
    """Mean Absolute Scaled Error"""
    naive_forecast = _naive_forecasting(actual, seasonality)
    return np.mean(np.abs(_error(actual, predicted))) / np.mean(np.abs(_error(actual[seasonality:], naive_forecast[seasonality:])))

# Function to evaluate all metrics for the forecasts of a given variable
def evaluate_forecasts(df, forecast_horizon):
    variables = df['variable'].unique()
    results = []

    for variable in variables:
        actual = df[df['variable'] == variable]['Actual'].values
        naive_forecast = _naive_forecasting(actual)  # naive forecast for comparison

        for model in [
                      'var',
                      'tvar',
                      'VES',
                      'Nbeatsx',
                      'LGBM',
                      'DTS_CB',
                      'DTS_XGB',
                      'NHITS',
                      'TSMixer',
                      'BRNN',
                      'XGB',
                      'CatBoost',
                      'szbvar_finetuned',
                      'TiDE',
                      'TFT'
                      ]:
            forecast = df[df['variable'] == variable][model].values

            # Append results for each model
            results.append({
                'Variable': variable,
                'Model': model,
                'Horizon': forecast_horizon,
                'RMSE': rmse(actual, forecast),
                'SMAPE': smape(actual, forecast),
                'MDRAE': mdrae(actual, forecast, naive_forecast),
                'MDAPE': mdape(actual, forecast),
                'Theil\'s U1': theils_u1(forecast, actual),
                'MASE': mase(actual, forecast),
                'Theil\'s U2': theils_u2(forecast, actual)
            })

    return pd.DataFrame(results)

# Step 1: Load the forecast data files
forecast_12m_file_path = '/content/Forecasts_Datasets/Forecasts_CANADA_12M.csv'
forecast_24m_file_path = '/content/Forecasts_Datasets/Forecasts_CANADA_24M.csv'

forecast_12m_df = pd.read_csv(forecast_12m_file_path)
forecast_24m_df = pd.read_csv(forecast_24m_file_path)

# Step 2: Evaluate the forecasts for both horizons
performance_12m = evaluate_forecasts(forecast_12m_df, '12M')
performance_24m = evaluate_forecasts(forecast_24m_df, '24M')

# Step 3: Combine results and write to a CSV file
performance_combined = pd.concat([performance_12m, performance_24m])
output_file_path = '/content/Forecasts_Performance_Eval_results/canada_forecast_performance_metrics_12M_24M_revised_20260429.csv'
performance_combined.to_csv(output_file_path, index=False)

print(f'Performance metrics saved to {output_file_path}')

Performance metrics saved to /content/Forecasts_Performance_Eval_results/canada_forecast_performance_metrics_12M_24M_revised_20260429.csv


**Forecasts Performance Evaluation**

- **Country: USA**

In [6]:
import pandas as pd
import numpy as np
from sklearn import metrics
from sklearn.metrics import mean_squared_log_error

EPSILON = 1e-10

# Helper functions for error calculations
def _error(actual: np.ndarray, predicted: np.ndarray):
    """Simple error"""
    return actual - predicted

def _percentage_error(actual: np.ndarray, predicted: np.ndarray):
    """Percentage error (not multiplied by 100)"""
    return _error(actual, predicted) / (actual + EPSILON)

def _naive_forecasting(actual: np.ndarray, seasonality: int = 1):
    """Naive forecasting method which just repeats previous samples"""
    if len(actual) <= seasonality:
        return np.full_like(actual, np.nan)  # Handle cases where not enough data for naive forecasting
    return np.concatenate((np.full(seasonality, actual[0]), actual[:-seasonality]))

# Function for RMSE
def rmse(actual: np.ndarray, predicted: np.ndarray):
    """Root Mean Squared Error"""
    return np.sqrt(np.mean(np.square(_error(actual, predicted))))

# Function for SMAPE
def smape(actual: np.ndarray, predicted: np.ndarray):
    """Symmetric Mean Absolute Percentage Error"""
    return np.mean(2.0 * np.abs(actual - predicted) / (np.abs(actual) + np.abs(predicted) + EPSILON))

# Function for MDRAE
def mdrae(actual: np.ndarray, predicted: np.ndarray, benchmark: np.ndarray = None):
    """Median Relative Absolute Error"""
    if benchmark is None:
        benchmark = _naive_forecasting(actual)
    abs_error_model = np.abs(predicted - actual)
    abs_error_naive = np.abs(benchmark - actual)
    return np.median(abs_error_model / (abs_error_naive + EPSILON))

# Function for MDAPE
def mdape(actual: np.ndarray, predicted: np.ndarray):
    """Median Absolute Percentage Error"""
    return np.median(np.abs(_percentage_error(actual, predicted))) * 100

# Updated Theil's U1 function
def theils_u1(f: np.ndarray, y: np.ndarray):
    """Theil's U1 statistic"""
    df = pd.DataFrame({'f_i': f, 'y_i': y})
    df['(f_i - y_i)^2'] = np.square(df['f_i'] - df['y_i'])
    df['y_i^2'] = np.square(df['y_i'])
    df['f_i^2'] = np.square(df['f_i'])
    return np.sqrt(np.mean(df['(f_i - y_i)^2'])) / (np.sqrt(np.mean(df['y_i^2'])) + np.sqrt(np.mean(df['f_i^2'])))

# Theil's U2 function
def theils_u2(f: np.ndarray, y: np.ndarray) -> float:
    """Theil's U2 statistic"""
    df = pd.DataFrame({'f_i+1': f.flatten(), 'y_i+1': y.flatten()})
    df['y_i'] = df['y_i+1'].shift(1)
    df['numerator'] = np.square((df['f_i+1'] - df['y_i+1']) / df['y_i'])
    df['denominator'] = np.square((df['y_i+1'] - df['y_i']) / df['y_i'])
    df.dropna(inplace=True)
    return np.sqrt(np.sum(df['numerator']) / np.sum(df['denominator']))

# Function for MASE
def mase(actual: np.ndarray, predicted: np.ndarray, seasonality: int = 1):
    """Mean Absolute Scaled Error"""
    naive_forecast = _naive_forecasting(actual, seasonality)
    return np.mean(np.abs(_error(actual, predicted))) / np.mean(np.abs(_error(actual[seasonality:], naive_forecast[seasonality:])))

# Function to evaluate all metrics for the forecasts of a given variable
def evaluate_forecasts(df, forecast_horizon):
    variables = df['variable'].unique()
    results = []

    for variable in variables:
        actual = df[df['variable'] == variable]['Actual'].values
        naive_forecast = _naive_forecasting(actual)  # naive forecast for comparison

        for model in [
                      'var',
                      'tvar',
                      'VES',
                      'Nbeatsx',
                      'LGBM',
                      'DTS_CB',
                      'DTS_XGB',
                      'NHITS',
                      'TSMixer',
                      'BRNN',
                      'XGB',
                      'CatBoost',
                      'szbvar_finetuned',
                      'TiDE',
                      'TFT'
                      ]:
            forecast = df[df['variable'] == variable][model].values

            # Append results for each model
            results.append({
                'Variable': variable,
                'Model': model,
                'Horizon': forecast_horizon,
                'RMSE': rmse(actual, forecast),
                'SMAPE': smape(actual, forecast),
                'MDRAE': mdrae(actual, forecast, naive_forecast),
                'MDAPE': mdape(actual, forecast),
                'Theil\'s U1': theils_u1(forecast, actual),
                'MASE': mase(actual, forecast),
                'Theil\'s U2': theils_u2(forecast, actual)
            })

    return pd.DataFrame(results)

# Step 1: Load the forecast data files
forecast_12m_file_path = '/content/Forecasts_Datasets/Forecasts_USA_12M.csv'
forecast_24m_file_path = '/content/Forecasts_Datasets/Forecasts_USA_24M.csv'

forecast_12m_df = pd.read_csv(forecast_12m_file_path)
forecast_24m_df = pd.read_csv(forecast_24m_file_path)

# Step 2: Evaluate the forecasts for both horizons
performance_12m = evaluate_forecasts(forecast_12m_df, '12M')
performance_24m = evaluate_forecasts(forecast_24m_df, '24M')

# Step 3: Combine results and write to a CSV file
performance_combined = pd.concat([performance_12m, performance_24m])
output_file_path = '/content/Forecasts_Performance_Eval_results/usa_forecast_performance_metrics_12M_24M_revised_20260429.csv'
performance_combined.to_csv(output_file_path, index=False)

print(f'Performance metrics saved to {output_file_path}')

Performance metrics saved to /content/Forecasts_Performance_Eval_results/usa_forecast_performance_metrics_12M_24M_revised_20260429.csv


**Forecasts Performance Evaluation**

- **Country: France**

In [7]:
import pandas as pd
import numpy as np
from sklearn import metrics
from sklearn.metrics import mean_squared_log_error

EPSILON = 1e-10

# Helper functions for error calculations
def _error(actual: np.ndarray, predicted: np.ndarray):
    """Simple error"""
    return actual - predicted

def _percentage_error(actual: np.ndarray, predicted: np.ndarray):
    """Percentage error (not multiplied by 100)"""
    return _error(actual, predicted) / (actual + EPSILON)

def _naive_forecasting(actual: np.ndarray, seasonality: int = 1):
    """Naive forecasting method which just repeats previous samples"""
    if len(actual) <= seasonality:
        return np.full_like(actual, np.nan)  # Handle cases where not enough data for naive forecasting
    return np.concatenate((np.full(seasonality, actual[0]), actual[:-seasonality]))

# Function for RMSE
def rmse(actual: np.ndarray, predicted: np.ndarray):
    """Root Mean Squared Error"""
    return np.sqrt(np.mean(np.square(_error(actual, predicted))))

# Function for SMAPE
def smape(actual: np.ndarray, predicted: np.ndarray):
    """Symmetric Mean Absolute Percentage Error"""
    return np.mean(2.0 * np.abs(actual - predicted) / (np.abs(actual) + np.abs(predicted) + EPSILON))

# Function for MDRAE
def mdrae(actual: np.ndarray, predicted: np.ndarray, benchmark: np.ndarray = None):
    """Median Relative Absolute Error"""
    if benchmark is None:
        benchmark = _naive_forecasting(actual)
    abs_error_model = np.abs(predicted - actual)
    abs_error_naive = np.abs(benchmark - actual)
    return np.median(abs_error_model / (abs_error_naive + EPSILON))

# Function for MDAPE
def mdape(actual: np.ndarray, predicted: np.ndarray):
    """Median Absolute Percentage Error"""
    return np.median(np.abs(_percentage_error(actual, predicted))) * 100

# Updated Theil's U1 function
def theils_u1(f: np.ndarray, y: np.ndarray):
    """Theil's U1 statistic"""
    df = pd.DataFrame({'f_i': f, 'y_i': y})
    df['(f_i - y_i)^2'] = np.square(df['f_i'] - df['y_i'])
    df['y_i^2'] = np.square(df['y_i'])
    df['f_i^2'] = np.square(df['f_i'])
    return np.sqrt(np.mean(df['(f_i - y_i)^2'])) / (np.sqrt(np.mean(df['y_i^2'])) + np.sqrt(np.mean(df['f_i^2'])))

# Theil's U2 function
def theils_u2(f: np.ndarray, y: np.ndarray) -> float:
    """Theil's U2 statistic"""
    df = pd.DataFrame({'f_i+1': f.flatten(), 'y_i+1': y.flatten()})
    df['y_i'] = df['y_i+1'].shift(1)
    df['numerator'] = np.square((df['f_i+1'] - df['y_i+1']) / df['y_i'])
    df['denominator'] = np.square((df['y_i+1'] - df['y_i']) / df['y_i'])
    df.dropna(inplace=True)
    return np.sqrt(np.sum(df['numerator']) / np.sum(df['denominator']))

# Function for MASE
def mase(actual: np.ndarray, predicted: np.ndarray, seasonality: int = 1):
    """Mean Absolute Scaled Error"""
    naive_forecast = _naive_forecasting(actual, seasonality)
    return np.mean(np.abs(_error(actual, predicted))) / np.mean(np.abs(_error(actual[seasonality:], naive_forecast[seasonality:])))

# Function to evaluate all metrics for the forecasts of a given variable
def evaluate_forecasts(df, forecast_horizon):
    variables = df['variable'].unique()
    results = []

    for variable in variables:
        actual = df[df['variable'] == variable]['Actual'].values
        naive_forecast = _naive_forecasting(actual)  # naive forecast for comparison

        for model in [
                      'var',
                      'tvar',
                      'VES',
                      'Nbeatsx',
                      'LGBM',
                      'DTS_CB',
                      'DTS_XGB',
                      'NHITS',
                      'TSMixer',
                      'BRNN',
                      'XGB',
                      'CatBoost',
                      'szbvar_finetuned',
                      'TiDE',
                      'TFT'
                      ]:
            forecast = df[df['variable'] == variable][model].values

            # Append results for each model
            results.append({
                'Variable': variable,
                'Model': model,
                'Horizon': forecast_horizon,
                'RMSE': rmse(actual, forecast),
                'SMAPE': smape(actual, forecast),
                'MDRAE': mdrae(actual, forecast, naive_forecast),
                'MDAPE': mdape(actual, forecast),
                'Theil\'s U1': theils_u1(forecast, actual),
                'MASE': mase(actual, forecast),
                'Theil\'s U2': theils_u2(forecast, actual)
            })

    return pd.DataFrame(results)

# Step 1: Load the forecast data files
forecast_12m_file_path = '/content/Forecasts_Datasets/Forecasts_FRANCE_12M.csv'
forecast_24m_file_path = '/content/Forecasts_Datasets/Forecasts_FRANCE_24M.csv'

forecast_12m_df = pd.read_csv(forecast_12m_file_path)
forecast_24m_df = pd.read_csv(forecast_24m_file_path)

# Step 2: Evaluate the forecasts for both horizons
performance_12m = evaluate_forecasts(forecast_12m_df, '12M')
performance_24m = evaluate_forecasts(forecast_24m_df, '24M')

# Step 3: Combine results and write to a CSV file
performance_combined = pd.concat([performance_12m, performance_24m])
output_file_path = '/content/Forecasts_Performance_Eval_results/france_forecast_performance_metrics_12M_24M_revised_20260429.csv'
performance_combined.to_csv(output_file_path, index=False)

print(f'Performance metrics saved to {output_file_path}')

Performance metrics saved to /content/Forecasts_Performance_Eval_results/france_forecast_performance_metrics_12M_24M_revised_20260429.csv


**Forecasts Performance Evaluation**

- **Country: Germany**

In [8]:
import pandas as pd
import numpy as np
from sklearn import metrics
from sklearn.metrics import mean_squared_log_error

EPSILON = 1e-10

# Helper functions for error calculations
def _error(actual: np.ndarray, predicted: np.ndarray):
    """Simple error"""
    return actual - predicted

def _percentage_error(actual: np.ndarray, predicted: np.ndarray):
    """Percentage error (not multiplied by 100)"""
    return _error(actual, predicted) / (actual + EPSILON)

def _naive_forecasting(actual: np.ndarray, seasonality: int = 1):
    """Naive forecasting method which just repeats previous samples"""
    if len(actual) <= seasonality:
        return np.full_like(actual, np.nan)  # Handle cases where not enough data for naive forecasting
    return np.concatenate((np.full(seasonality, actual[0]), actual[:-seasonality]))

# Function for RMSE
def rmse(actual: np.ndarray, predicted: np.ndarray):
    """Root Mean Squared Error"""
    return np.sqrt(np.mean(np.square(_error(actual, predicted))))

# Function for SMAPE
def smape(actual: np.ndarray, predicted: np.ndarray):
    """Symmetric Mean Absolute Percentage Error"""
    return np.mean(2.0 * np.abs(actual - predicted) / (np.abs(actual) + np.abs(predicted) + EPSILON))

# Function for MDRAE
def mdrae(actual: np.ndarray, predicted: np.ndarray, benchmark: np.ndarray = None):
    """Median Relative Absolute Error"""
    if benchmark is None:
        benchmark = _naive_forecasting(actual)
    abs_error_model = np.abs(predicted - actual)
    abs_error_naive = np.abs(benchmark - actual)
    return np.median(abs_error_model / (abs_error_naive + EPSILON))

# Function for MDAPE
def mdape(actual: np.ndarray, predicted: np.ndarray):
    """Median Absolute Percentage Error"""
    return np.median(np.abs(_percentage_error(actual, predicted))) * 100

# Updated Theil's U1 function
def theils_u1(f: np.ndarray, y: np.ndarray):
    """Theil's U1 statistic"""
    df = pd.DataFrame({'f_i': f, 'y_i': y})
    df['(f_i - y_i)^2'] = np.square(df['f_i'] - df['y_i'])
    df['y_i^2'] = np.square(df['y_i'])
    df['f_i^2'] = np.square(df['f_i'])
    return np.sqrt(np.mean(df['(f_i - y_i)^2'])) / (np.sqrt(np.mean(df['y_i^2'])) + np.sqrt(np.mean(df['f_i^2'])))

# Theil's U2 function
def theils_u2(f: np.ndarray, y: np.ndarray) -> float:
    """Theil's U2 statistic"""
    df = pd.DataFrame({'f_i+1': f.flatten(), 'y_i+1': y.flatten()})
    df['y_i'] = df['y_i+1'].shift(1)
    df['numerator'] = np.square((df['f_i+1'] - df['y_i+1']) / df['y_i'])
    df['denominator'] = np.square((df['y_i+1'] - df['y_i']) / df['y_i'])
    df.dropna(inplace=True)
    return np.sqrt(np.sum(df['numerator']) / np.sum(df['denominator']))

# Function for MASE
def mase(actual: np.ndarray, predicted: np.ndarray, seasonality: int = 1):
    """Mean Absolute Scaled Error"""
    naive_forecast = _naive_forecasting(actual, seasonality)
    return np.mean(np.abs(_error(actual, predicted))) / np.mean(np.abs(_error(actual[seasonality:], naive_forecast[seasonality:])))

# Function to evaluate all metrics for the forecasts of a given variable
def evaluate_forecasts(df, forecast_horizon):
    variables = df['variable'].unique()
    results = []

    for variable in variables:
        actual = df[df['variable'] == variable]['Actual'].values
        naive_forecast = _naive_forecasting(actual)  # naive forecast for comparison

        for model in [
                      'var',
                      'tvar',
                      'VES',
                      'Nbeatsx',
                      'LGBM',
                      'DTS_CB',
                      'DTS_XGB',
                      'NHITS',
                      'TSMixer',
                      'BRNN',
                      'XGB',
                      'CatBoost',
                      'szbvar_finetuned',
                      'TiDE',
                      'TFT'
                      ]:
            forecast = df[df['variable'] == variable][model].values

            # Append results for each model
            results.append({
                'Variable': variable,
                'Model': model,
                'Horizon': forecast_horizon,
                'RMSE': rmse(actual, forecast),
                'SMAPE': smape(actual, forecast),
                'MDRAE': mdrae(actual, forecast, naive_forecast),
                'MDAPE': mdape(actual, forecast),
                'Theil\'s U1': theils_u1(forecast, actual),
                'MASE': mase(actual, forecast),
                'Theil\'s U2': theils_u2(forecast, actual)
            })

    return pd.DataFrame(results)

# Step 1: Load the forecast data files
forecast_12m_file_path = '/content/Forecasts_Datasets/Forecasts_GERMANY_12M.csv'
forecast_24m_file_path = '/content/Forecasts_Datasets/Forecasts_GERMANY_24M.csv'

forecast_12m_df = pd.read_csv(forecast_12m_file_path)
forecast_24m_df = pd.read_csv(forecast_24m_file_path)

# Step 2: Evaluate the forecasts for both horizons
performance_12m = evaluate_forecasts(forecast_12m_df, '12M')
performance_24m = evaluate_forecasts(forecast_24m_df, '24M')

# Step 3: Combine results and write to a CSV file
performance_combined = pd.concat([performance_12m, performance_24m])
output_file_path = '/content/Forecasts_Performance_Eval_results/germany_forecast_performance_metrics_12M_24M_revised_20260429.csv'
performance_combined.to_csv(output_file_path, index=False)

print(f'Performance metrics saved to {output_file_path}')

Performance metrics saved to /content/Forecasts_Performance_Eval_results/germany_forecast_performance_metrics_12M_24M_revised_20260429.csv


**Forecasts Performance Evaluation**

- **Country: Japan**

In [9]:
import pandas as pd
import numpy as np
from sklearn import metrics
from sklearn.metrics import mean_squared_log_error

EPSILON = 1e-10

# Helper functions for error calculations
def _error(actual: np.ndarray, predicted: np.ndarray):
    """Simple error"""
    return actual - predicted

def _percentage_error(actual: np.ndarray, predicted: np.ndarray):
    """Percentage error (not multiplied by 100)"""
    return _error(actual, predicted) / (actual + EPSILON)

def _naive_forecasting(actual: np.ndarray, seasonality: int = 1):
    """Naive forecasting method which just repeats previous samples"""
    if len(actual) <= seasonality:
        return np.full_like(actual, np.nan)  # Handle cases where not enough data for naive forecasting
    return np.concatenate((np.full(seasonality, actual[0]), actual[:-seasonality]))

# Function for RMSE
def rmse(actual: np.ndarray, predicted: np.ndarray):
    """Root Mean Squared Error"""
    return np.sqrt(np.mean(np.square(_error(actual, predicted))))

# Function for SMAPE
def smape(actual: np.ndarray, predicted: np.ndarray):
    """Symmetric Mean Absolute Percentage Error"""
    return np.mean(2.0 * np.abs(actual - predicted) / (np.abs(actual) + np.abs(predicted) + EPSILON))

# Function for MDRAE
def mdrae(actual: np.ndarray, predicted: np.ndarray, benchmark: np.ndarray = None):
    """Median Relative Absolute Error"""
    if benchmark is None:
        benchmark = _naive_forecasting(actual)
    abs_error_model = np.abs(predicted - actual)
    abs_error_naive = np.abs(benchmark - actual)
    return np.median(abs_error_model / (abs_error_naive + EPSILON))

# Function for MDAPE
def mdape(actual: np.ndarray, predicted: np.ndarray):
    """Median Absolute Percentage Error"""
    return np.median(np.abs(_percentage_error(actual, predicted))) * 100

# Updated Theil's U1 function
def theils_u1(f: np.ndarray, y: np.ndarray):
    """Theil's U1 statistic"""
    df = pd.DataFrame({'f_i': f, 'y_i': y})
    df['(f_i - y_i)^2'] = np.square(df['f_i'] - df['y_i'])
    df['y_i^2'] = np.square(df['y_i'])
    df['f_i^2'] = np.square(df['f_i'])
    return np.sqrt(np.mean(df['(f_i - y_i)^2'])) / (np.sqrt(np.mean(df['y_i^2'])) + np.sqrt(np.mean(df['f_i^2'])))

# Theil's U2 function
def theils_u2(f: np.ndarray, y: np.ndarray) -> float:
    """Theil's U2 statistic"""
    df = pd.DataFrame({'f_i+1': f.flatten(), 'y_i+1': y.flatten()})
    df['y_i'] = df['y_i+1'].shift(1)
    df['numerator'] = np.square((df['f_i+1'] - df['y_i+1']) / df['y_i'])
    df['denominator'] = np.square((df['y_i+1'] - df['y_i']) / df['y_i'])
    df.dropna(inplace=True)
    return np.sqrt(np.sum(df['numerator']) / np.sum(df['denominator']))

# Function for MASE
def mase(actual: np.ndarray, predicted: np.ndarray, seasonality: int = 1):
    """Mean Absolute Scaled Error"""
    naive_forecast = _naive_forecasting(actual, seasonality)
    return np.mean(np.abs(_error(actual, predicted))) / np.mean(np.abs(_error(actual[seasonality:], naive_forecast[seasonality:])))

# Function to evaluate all metrics for the forecasts of a given variable
def evaluate_forecasts(df, forecast_horizon):
    variables = df['variable'].unique()
    results = []

    for variable in variables:
        actual = df[df['variable'] == variable]['Actual'].values
        naive_forecast = _naive_forecasting(actual)  # naive forecast for comparison

        for model in [
                      'var',
                      'tvar',
                      'VES',
                      'Nbeatsx',
                      'LGBM',
                      'DTS_CB',
                      'DTS_XGB',
                      'NHITS',
                      'TSMixer',
                      'BRNN',
                      'XGB',
                      'CatBoost',
                      'szbvar_finetuned',
                      'TiDE',
                      'TFT'
                      ]:
            forecast = df[df['variable'] == variable][model].values

            # Append results for each model
            results.append({
                'Variable': variable,
                'Model': model,
                'Horizon': forecast_horizon,
                'RMSE': rmse(actual, forecast),
                'SMAPE': smape(actual, forecast),
                'MDRAE': mdrae(actual, forecast, naive_forecast),
                'MDAPE': mdape(actual, forecast),
                'Theil\'s U1': theils_u1(forecast, actual),
                'MASE': mase(actual, forecast),
                'Theil\'s U2': theils_u2(forecast, actual)
            })

    return pd.DataFrame(results)

# Step 1: Load the forecast data files
forecast_12m_file_path = '/content/Forecasts_Datasets/Forecasts_JAPAN_12M.csv'
forecast_24m_file_path = '/content/Forecasts_Datasets/Forecasts_JAPAN_24M.csv'

forecast_12m_df = pd.read_csv(forecast_12m_file_path)
forecast_24m_df = pd.read_csv(forecast_24m_file_path)

# Step 2: Evaluate the forecasts for both horizons
performance_12m = evaluate_forecasts(forecast_12m_df, '12M')
performance_24m = evaluate_forecasts(forecast_24m_df, '24M')

# Step 3: Combine results and write to a CSV file
performance_combined = pd.concat([performance_12m, performance_24m])
output_file_path = '/content/Forecasts_Performance_Eval_results/japan_forecast_performance_metrics_12M_24M_revised_20260429.csv'
performance_combined.to_csv(output_file_path, index=False)

print(f'Performance metrics saved to {output_file_path}')

Performance metrics saved to /content/Forecasts_Performance_Eval_results/japan_forecast_performance_metrics_12M_24M_revised_20260429.csv


**Forecasts Performance Evaluation**

- **Country: UK**

In [10]:
import pandas as pd
import numpy as np
from sklearn import metrics
from sklearn.metrics import mean_squared_log_error

EPSILON = 1e-10

# Helper functions for error calculations
def _error(actual: np.ndarray, predicted: np.ndarray):
    """Simple error"""
    return actual - predicted

def _percentage_error(actual: np.ndarray, predicted: np.ndarray):
    """Percentage error (not multiplied by 100)"""
    return _error(actual, predicted) / (actual + EPSILON)

def _naive_forecasting(actual: np.ndarray, seasonality: int = 1):
    """Naive forecasting method which just repeats previous samples"""
    if len(actual) <= seasonality:
        return np.full_like(actual, np.nan)  # Handle cases where not enough data for naive forecasting
    return np.concatenate((np.full(seasonality, actual[0]), actual[:-seasonality]))

# Function for RMSE
def rmse(actual: np.ndarray, predicted: np.ndarray):
    """Root Mean Squared Error"""
    return np.sqrt(np.mean(np.square(_error(actual, predicted))))

# Function for SMAPE
def smape(actual: np.ndarray, predicted: np.ndarray):
    """Symmetric Mean Absolute Percentage Error"""
    return np.mean(2.0 * np.abs(actual - predicted) / (np.abs(actual) + np.abs(predicted) + EPSILON))

# Function for MDRAE
def mdrae(actual: np.ndarray, predicted: np.ndarray, benchmark: np.ndarray = None):
    """Median Relative Absolute Error"""
    if benchmark is None:
        benchmark = _naive_forecasting(actual)
    abs_error_model = np.abs(predicted - actual)
    abs_error_naive = np.abs(benchmark - actual)
    return np.median(abs_error_model / (abs_error_naive + EPSILON))

# Function for MDAPE
def mdape(actual: np.ndarray, predicted: np.ndarray):
    """Median Absolute Percentage Error"""
    return np.median(np.abs(_percentage_error(actual, predicted))) * 100

# Updated Theil's U1 function
def theils_u1(f: np.ndarray, y: np.ndarray):
    """Theil's U1 statistic"""
    df = pd.DataFrame({'f_i': f, 'y_i': y})
    df['(f_i - y_i)^2'] = np.square(df['f_i'] - df['y_i'])
    df['y_i^2'] = np.square(df['y_i'])
    df['f_i^2'] = np.square(df['f_i'])
    return np.sqrt(np.mean(df['(f_i - y_i)^2'])) / (np.sqrt(np.mean(df['y_i^2'])) + np.sqrt(np.mean(df['f_i^2'])))

# Theil's U2 function
def theils_u2(f: np.ndarray, y: np.ndarray) -> float:
    """Theil's U2 statistic"""
    df = pd.DataFrame({'f_i+1': f.flatten(), 'y_i+1': y.flatten()})
    df['y_i'] = df['y_i+1'].shift(1)
    df['numerator'] = np.square((df['f_i+1'] - df['y_i+1']) / df['y_i'])
    df['denominator'] = np.square((df['y_i+1'] - df['y_i']) / df['y_i'])
    df.dropna(inplace=True)
    return np.sqrt(np.sum(df['numerator']) / np.sum(df['denominator']))

# Function for MASE
def mase(actual: np.ndarray, predicted: np.ndarray, seasonality: int = 1):
    """Mean Absolute Scaled Error"""
    naive_forecast = _naive_forecasting(actual, seasonality)
    return np.mean(np.abs(_error(actual, predicted))) / np.mean(np.abs(_error(actual[seasonality:], naive_forecast[seasonality:])))

# Function to evaluate all metrics for the forecasts of a given variable
def evaluate_forecasts(df, forecast_horizon):
    variables = df['variable'].unique()
    results = []

    for variable in variables:
        actual = df[df['variable'] == variable]['Actual'].values
        naive_forecast = _naive_forecasting(actual)  # naive forecast for comparison

        for model in [
                      'var',
                      'tvar',
                      'VES',
                      'Nbeatsx',
                      'LGBM',
                      'DTS_CB',
                      'DTS_XGB',
                      'NHITS',
                      'TSMixer',
                      'BRNN',
                      'XGB',
                      'CatBoost',
                      'szbvar_finetuned',
                      'TiDE',
                      'TFT'
                      ]:
            forecast = df[df['variable'] == variable][model].values

            # Append results for each model
            results.append({
                'Variable': variable,
                'Model': model,
                'Horizon': forecast_horizon,
                'RMSE': rmse(actual, forecast),
                'SMAPE': smape(actual, forecast),
                'MDRAE': mdrae(actual, forecast, naive_forecast),
                'MDAPE': mdape(actual, forecast),
                'Theil\'s U1': theils_u1(forecast, actual),
                'MASE': mase(actual, forecast),
                'Theil\'s U2': theils_u2(forecast, actual)
            })

    return pd.DataFrame(results)

# Step 1: Load the forecast data files
forecast_12m_file_path = '/content/Forecasts_Datasets/Forecasts_UK_12M.csv'
forecast_24m_file_path = '/content/Forecasts_Datasets/Forecasts_UK_24M.csv'

forecast_12m_df = pd.read_csv(forecast_12m_file_path)
forecast_24m_df = pd.read_csv(forecast_24m_file_path)

# Step 2: Evaluate the forecasts for both horizons
performance_12m = evaluate_forecasts(forecast_12m_df, '12M')
performance_24m = evaluate_forecasts(forecast_24m_df, '24M')

# Step 3: Combine results and write to a CSV file
performance_combined = pd.concat([performance_12m, performance_24m])
output_file_path = '/content/Forecasts_Performance_Eval_results/uk_forecast_performance_metrics_12M_24M_revised_20260429.csv'
performance_combined.to_csv(output_file_path, index=False)

print(f'Performance metrics saved to {output_file_path}')

Performance metrics saved to /content/Forecasts_Performance_Eval_results/uk_forecast_performance_metrics_12M_24M_revised_20260429.csv


**Forecasts Performance Evaluation**

- **Country: Italy**

In [11]:
import pandas as pd
import numpy as np
from sklearn import metrics
from sklearn.metrics import mean_squared_log_error

EPSILON = 1e-10

# Helper functions for error calculations
def _error(actual: np.ndarray, predicted: np.ndarray):
    """Simple error"""
    return actual - predicted

def _percentage_error(actual: np.ndarray, predicted: np.ndarray):
    """Percentage error (not multiplied by 100)"""
    return _error(actual, predicted) / (actual + EPSILON)

def _naive_forecasting(actual: np.ndarray, seasonality: int = 1):
    """Naive forecasting method which just repeats previous samples"""
    if len(actual) <= seasonality:
        return np.full_like(actual, np.nan)  # Handle cases where not enough data for naive forecasting
    return np.concatenate((np.full(seasonality, actual[0]), actual[:-seasonality]))

# Function for RMSE
def rmse(actual: np.ndarray, predicted: np.ndarray):
    """Root Mean Squared Error"""
    return np.sqrt(np.mean(np.square(_error(actual, predicted))))

# Function for SMAPE
def smape(actual: np.ndarray, predicted: np.ndarray):
    """Symmetric Mean Absolute Percentage Error"""
    return np.mean(2.0 * np.abs(actual - predicted) / (np.abs(actual) + np.abs(predicted) + EPSILON))

# Function for MDRAE
def mdrae(actual: np.ndarray, predicted: np.ndarray, benchmark: np.ndarray = None):
    """Median Relative Absolute Error"""
    if benchmark is None:
        benchmark = _naive_forecasting(actual)
    abs_error_model = np.abs(predicted - actual)
    abs_error_naive = np.abs(benchmark - actual)
    return np.median(abs_error_model / (abs_error_naive + EPSILON))

# Function for MDAPE
def mdape(actual: np.ndarray, predicted: np.ndarray):
    """Median Absolute Percentage Error"""
    return np.median(np.abs(_percentage_error(actual, predicted))) * 100

# Updated Theil's U1 function
def theils_u1(f: np.ndarray, y: np.ndarray):
    """Theil's U1 statistic"""
    df = pd.DataFrame({'f_i': f, 'y_i': y})
    df['(f_i - y_i)^2'] = np.square(df['f_i'] - df['y_i'])
    df['y_i^2'] = np.square(df['y_i'])
    df['f_i^2'] = np.square(df['f_i'])
    return np.sqrt(np.mean(df['(f_i - y_i)^2'])) / (np.sqrt(np.mean(df['y_i^2'])) + np.sqrt(np.mean(df['f_i^2'])))

# Theil's U2 function
def theils_u2(f: np.ndarray, y: np.ndarray) -> float:
    """Theil's U2 statistic"""
    df = pd.DataFrame({'f_i+1': f.flatten(), 'y_i+1': y.flatten()})
    df['y_i'] = df['y_i+1'].shift(1)
    df['numerator'] = np.square((df['f_i+1'] - df['y_i+1']) / df['y_i'])
    df['denominator'] = np.square((df['y_i+1'] - df['y_i']) / df['y_i'])
    df.dropna(inplace=True)
    return np.sqrt(np.sum(df['numerator']) / np.sum(df['denominator']))

# Function for MASE
def mase(actual: np.ndarray, predicted: np.ndarray, seasonality: int = 1):
    """Mean Absolute Scaled Error"""
    naive_forecast = _naive_forecasting(actual, seasonality)
    return np.mean(np.abs(_error(actual, predicted))) / np.mean(np.abs(_error(actual[seasonality:], naive_forecast[seasonality:])))

# Function to evaluate all metrics for the forecasts of a given variable
def evaluate_forecasts(df, forecast_horizon):
    variables = df['variable'].unique()
    results = []

    for variable in variables:
        actual = df[df['variable'] == variable]['Actual'].values
        naive_forecast = _naive_forecasting(actual)  # naive forecast for comparison

        for model in [
                      'var',
                      'tvar',
                      'VES',
                      'Nbeatsx',
                      'LGBM',
                      'DTS_CB',
                      'DTS_XGB',
                      'NHITS',
                      'TSMixer',
                      'BRNN',
                      'XGB',
                      'CatBoost',
                      'szbvar_finetuned',
                      'TiDE',
                      'TFT'
                      ]:
            forecast = df[df['variable'] == variable][model].values

            # Append results for each model
            results.append({
                'Variable': variable,
                'Model': model,
                'Horizon': forecast_horizon,
                'RMSE': rmse(actual, forecast),
                'SMAPE': smape(actual, forecast),
                'MDRAE': mdrae(actual, forecast, naive_forecast),
                'MDAPE': mdape(actual, forecast),
                'Theil\'s U1': theils_u1(forecast, actual),
                'MASE': mase(actual, forecast),
                'Theil\'s U2': theils_u2(forecast, actual)
            })

    return pd.DataFrame(results)

# Step 1: Load the forecast data files
forecast_12m_file_path = '/content/Forecasts_Datasets/Forecasts_ITALY_12M.csv'
forecast_24m_file_path = '/content/Forecasts_Datasets/Forecasts_ITALY_24M.csv'

forecast_12m_df = pd.read_csv(forecast_12m_file_path)
forecast_24m_df = pd.read_csv(forecast_24m_file_path)

# Step 2: Evaluate the forecasts for both horizons
performance_12m = evaluate_forecasts(forecast_12m_df, '12M')
performance_24m = evaluate_forecasts(forecast_24m_df, '24M')

# Step 3: Combine results and write to a CSV file
performance_combined = pd.concat([performance_12m, performance_24m])
output_file_path = '/content/Forecasts_Performance_Eval_results/italy_forecast_performance_metrics_12M_24M_revised_20260429.csv'
performance_combined.to_csv(output_file_path, index=False)

print(f'Performance metrics saved to {output_file_path}')

Performance metrics saved to /content/Forecasts_Performance_Eval_results/italy_forecast_performance_metrics_12M_24M_revised_20260429.csv
